
# Case EV vs Case Price - Statistical Appendix

Created: **2025-10-14 15:05:16Z**

This notebook produces the statistical results promised in the README:

- Stationarity checks (ADF)
- Granger causality: EV -> Price and Price -> EV
- Cointegration: Engle-Granger (Johansen optional)
- Basis half-life via AR(1)

It expects a tidy panel at `data/processed/ev_case_panel.csv` with columns:

```
date, case, ev_usd, case_price_usd, basis_usd, log_ev, log_price, dlog_ev, dlog_price, z_basis
```


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests
from statsmodels.tsa.api import VAR
import statsmodels.api as sm

# Paths (adjust if running outside the repo root)
DATA_ROOT = Path("data")
CLEAN = DATA_ROOT / "clean"
PROCESSED = DATA_ROOT / "processed"
STATS_DIR = PROCESSED / "stats"
PLOTS_DIR = Path("plots")

PROCESSED.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CSV_PANEL = PROCESSED / "ev_case_panel.csv"

print(f"Looking for panel CSV at: {CSV_PANEL.resolve()}")


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp312-cp312-win_amd64.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-win_amd64.whl.metadata (6.4 kB)
  Using cached pillow-11.3.0-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached pyparsing-3.2.5-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.1 MB 7.1 MB/s eta 0:00:01
   ----------- ---------------------------- 2.4/8.1 MB 7.1 MB/s eta 0:00:01
   --------------- ------------------------ 3.1/8.1 MB 5.6 MB/s eta 0:00:01
   ------------------- -------------------- 3.9/8.1 MB 5.3 MB/s eta 0:00:01
   --------------------------- ------------ 5.5/8.1 MB 5.6 MB/s eta 0:00:01
   ------------------------------------- --

In [3]:

assert CSV_PANEL.exists(), f"Missing {CSV_PANEL}. Please export the panel CSV before running."
df = pd.read_csv(CSV_PANEL, parse_dates=["date"])

# Basic checks
required = ["date","case","ev_usd","case_price_usd","basis_usd",
            "log_ev","log_price","dlog_ev","dlog_price","z_basis"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

df = df.sort_values(["case","date"]).reset_index(drop=True)
display(df.head())
print("Cases:", df['case'].nunique())


AssertionError: Missing data\processed\ev_case_panel.csv. Please export the panel CSV before running.

In [ ]:

def adf_pvalue(series, maxlag=None, regression='c', autolag='AIC'):
    series = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    if series.empty or series.nunique() < 3:
        return np.nan
    try:
        res = adfuller(series.values, maxlag=maxlag, regression=regression, autolag=autolag)
        return res[1]
    except Exception as e:
        return np.nan

def ar1_half_life(series):
    # Return half-life in periods for AR(1) on provided series (levels).
    # We estimate y_t = c + phi*y_{t-1} + eps. Half-life = -ln(2)/ln(phi) if |phi|<1.
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 10:
        return np.nan
    y = s.values[1:]
    x = s.values[:-1]
    X = sm.add_constant(x)
    try:
        model = sm.OLS(y, X, missing="drop").fit()
        phi = model.params[1]
        if np.abs(phi) >= 1:
            return np.inf
        return -np.log(2) / np.log(np.abs(phi))
    except Exception as e:
        return np.nan

def choose_var_lag(df2, maxlags=5):
    # Pick lag by AIC on a small VAR of [dlog_price, dlog_ev].
    m = df2[["dlog_price","dlog_ev"]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(m) < maxlags + 10:
        return 1
    best_p, best_aic = 1, np.inf
    for p in range(1, maxlags+1):
        try:
            model = VAR(m).fit(p)
            if model.aic < best_aic:
                best_aic, best_p = model.aic, p
        except Exception:
            continue
    return best_p if best_p>=1 else 1

def run_granger(df2, maxlags=5, verbose=False):
    # Return dict with both directions using dlog series, choosing lag by AIC.
    m = df2[["dlog_price","dlog_ev"]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(m) < maxlags + 10:
        return {"ev_to_price": None, "price_to_ev": None, "lags": None}

    p = choose_var_lag(df2, maxlags=maxlags)
    out = {"lags": p}

    # EV -> Price
    try:
        g = grangercausalitytests(m[["dlog_price","dlog_ev"]], maxlag=p, verbose=verbose)
        F, pval, _, _ = g[p][0]["ssr_ftest"]
        out["ev_to_price"] = {"F": float(F), "p": float(pval)}
    except Exception:
        out["ev_to_price"] = None

    # Price -> EV
    try:
        g = grangercausalitytests(m[["dlog_ev","dlog_price"]], maxlag=p, verbose=verbose)
        F, pval, _, _ = g[p][0]["ssr_ftest"]
        out["price_to_ev"] = {"F": float(F), "p": float(pval)}
    except Exception:
        out["price_to_ev"] = None

    return out

def run_cointegration(df2):
    # Engle-Granger on log levels: log_price ~ log_ev
    m = df2[["log_price","log_ev"]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(m) < 25:
        return None
    try:
        stat, pval, crit = coint(m["log_price"], m["log_ev"])
        return {"method": "engle-granger", "stat": float(stat), "p": float(pval), "crit": None}
    except Exception:
        return None


In [ ]:

cases = sorted(df["case"].unique())
rows = []

for c in cases:
    d = df[df["case"]==c].copy().sort_values("date")
    window = {"from": str(d["date"].min().date()), "to": str(d["date"].max().date())}
    n_obs = int(d.shape[0])

    # Stationarity (p-values)
    adf_price_p = adf_pvalue(d["log_price"])
    adf_ev_p    = adf_pvalue(d["log_ev"])
    adf_dprice_p= adf_pvalue(d["dlog_price"])
    adf_dev_p   = adf_pvalue(d["dlog_ev"])

    # Granger
    gr = run_granger(d, maxlags=5, verbose=False)

    # Cointegration
    ci = run_cointegration(d)

    # Basis half-life from z_basis (already standardized)
    hl = ar1_half_life(d["z_basis"])

    # Persist per-case JSON
    out = {
        "case": c,
        "window": window,
        "n_obs": n_obs,
        "stationarity": {
            "adf_p_log_price": adf_price_p,
            "adf_p_log_ev": adf_ev_p,
            "adf_p_dlog_price": adf_dprice_p,
            "adf_p_dlog_ev": adf_dev_p,
        },
        "granger": gr,
        "cointegration": ci,
        "basis_half_life_days": hl,
        "generated_at": pd.Timestamp.utcnow().isoformat() + "Z"
    }
    with open(STATS_DIR / (c.replace(" ", "_") + ".json"), "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    rows.append({
        "case": c,
        "n_obs": n_obs,
        "adf_p_log_price": adf_price_p,
        "adf_p_log_ev": adf_ev_p,
        "granger_lags": gr.get("lags"),
        "granger_p_ev_to_price": (gr.get("ev_to_price") or {}).get("p"),
        "granger_p_price_to_ev": (gr.get("price_to_ev") or {}).get("p"),
        "coint_p": (ci or {}).get("p") if ci else np.nan,
        "basis_half_life_days": hl
    })

summary = pd.DataFrame(rows).sort_values("case")
display(summary.head(10))
summary_path = PROCESSED / "stats_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path.resolve())


In [ ]:

# Simple showcase plots

# 1) Histogram of basis half-life
fig = plt.figure(figsize=(6,4))
summary["basis_half_life_days"].replace([np.inf], np.nan).dropna().hist(bins=20)
plt.title("Basis Half-Life (days)")
plt.xlabel("days"); plt.ylabel("count")
plt.tight_layout()
p1 = PLOTS_DIR / "basis_half_life_hist.png"
plt.savefig(p1, dpi=144)
plt.show()
print("Saved:", p1.resolve())

# 2) Scatter: Granger p-values (EV->Price vs Price->EV)
fig = plt.figure(figsize=(5,5))
x = summary["granger_p_ev_to_price"]
y = summary["granger_p_price_to_ev"]
plt.scatter(x, y, alpha=0.7)
plt.axvline(0.05, color="r", linestyle="--", lw=1)
plt.axhline(0.05, color="r", linestyle="--", lw=1)
plt.xlabel("p(EV -> Price)")
plt.ylabel("p(Price -> EV)")
plt.title("Granger p-values by case")
plt.tight_layout()
p2 = PLOTS_DIR / "granger_p_scatter.png"
plt.savefig(p2, dpi=144)
plt.show()
print("Saved:", p2.resolve())

# 3) Histogram of cointegration p-values
fig = plt.figure(figsize=(6,4))
summary["coint_p"].replace([np.inf], np.nan).dropna().hist(bins=20)
plt.title("Engle-Granger Cointegration p-values")
plt.xlabel("p-value"); plt.ylabel("count")
plt.tight_layout()
p3 = PLOTS_DIR / "cointegration_p_hist.png"
plt.savefig(p3, dpi=144)
plt.show()
print("Saved:", p3.resolve())
